# 📊 LIVR-Mini-Benchmark: GIAI ĐOẠN HUẤN LUYỆN TRÊN KAGGLE (IMPLEMENT MINI ON KAGGLE)

Chào mừng bạn đến với giai đoạn Huấn luyện của mô hình **LIVR (Latent Implicit Visual Reasoning)** trên môi trường Kaggle GPU. Notebook này được tinh chỉnh tối ưu đặc thù để huấn luyện hai giai đoạn (Stage 1 & Stage 2) một cách trơn tru, vượt qua các giới hạn bộ nhớ (VRAM OOM), tránh lỗi tràn số học (NaN Loss), và sửa lỗi lưu checkpoint rỗng (66KB) trên các dòng card GPU Tesla T4 hoặc P100 của Kaggle.

---

## 🧬 1. Ý tưởng cốt lõi của Kiến trúc LIVR

Mục tiêu của nghiên cứu **LIVR** là cải thiện khả năng **suy luận thị giác ẩn** của các mô hình đa phương thức lớn (MLLMs). Thông thường, khi đối mặt với một bức ảnh, các mô hình VLM truyền thống sẽ chú ý (attend) trực tiếp từ các câu hỏi ngôn ngữ sang các vùng ảnh thô. Cách làm này đôi khi khiến mô hình bị phân tâm bởi các thông tin nhiễu hoặc chỉ học vẹt mẫu từ ngữ.

LIVR đề xuất chèn thêm **$K = 16$ Latent Tokens** làm nút cổ chai thông tin (Visual Bottleneck) và phân tách quá trình học thành 2 giai đoạn:

1. **Stage 1 (Huấn luyện Nút Cổ Chai - Visual Bottlenecking)**:
   * **Cơ chế**: Chặn toàn bộ luồng chú ý trực tiếp từ Prompt (Câu hỏi) và Answer (Câu trả lời) đến ảnh gốc. Chỉ cho phép ảnh truyền thông tin vào $K$ Latent Tokens, và cho phép Prompt/Answer truy vấn thông tin từ Latent Tokens này.
   * **Mục tiêu**: Ép buộc 16 Latent Tokens học cách cô đọng, trừu tượng hóa và lưu trữ trọn vẹn đặc trưng thị giác cốt lõi của bức ảnh.
   * **Tham số cập nhật**: Chỉ cập nhật trọng số của **LoRA Adapters** và bảng biểu diễn vector của **Latent Embeddings**.

2. **Stage 2 (Huấn luyện Tích hợp - Joint Training)**:
   * **Cơ chế**: Mở lại luồng chú ý thông thường (Causal Attention). Mô hình lúc này được nhìn trực tiếp cả ảnh gốc và các Latent Tokens đã được huấn luyện tốt từ Stage 1.
   * **Mục tiêu**: Dạy mô hình cách phối hợp linh hoạt giữa các chi tiết thô của ảnh gốc và các biểu diễn ẩn trừu tượng trong Latent Tokens để tối ưu hóa khả năng suy luận cuối cùng.

```mermaid
graph TD
    subgraph Stage_1_Bit_Mat [Stage 1: Bịt mắt - Visual Bottleneck]
        Vision_1["Visual Tokens (Ảnh gốc)"] -->|Cho phép| Latent_1["Latent Tokens (K=16)"]
        Latent_1 -->|Truyền thông tin| Answer_1["Answer (Tính Loss)"]
        Vision_1 -.->|CHẶN ATTENTION (-30000.0)| Answer_1
    end
    
    subgraph Stage_2_Mo_Mat [Stage 2: Mở mắt - Joint Training]
        Vision_2["Visual Tokens (Ảnh gốc)"] -->|Mở lại Attention| Answer_2["Answer (Tính Loss)"]
        Latent_2["Latent Tokens (Đã học từ Stage 1)"] -->|Hỗ trợ suy luận| Answer_2
    end
```

---

## 🛠️ 2. Các điểm tối ưu hóa đặc thù trên Kaggle

*   **Giải quyết lỗi Checkpoint rỗng (66KB)**: Trong PyTorch, việc lọc `state_dict()` bằng thuộc tính `.requires_grad` trực tiếp trên các tensor trả về sẽ bị sai vì các tensor này đã bị ngắt đạo hàm. Chúng ta sửa đổi cơ chế lưu bằng cách đối chiếu tên tham số với `model.named_parameters()` để lọc chính xác các trọng số LoRA thực tế.
*   **Vá lỗi tương thích RoPE Index**: Qwen2.5-VL yêu cầu tính toán chỉ số vị trí xoay thông qua `get_rope_index`. Chúng ta sử dụng `inspect.signature` trong `src/mask_kaggle.py` để tương thích linh hoạt với các thay đổi signature của hàm này trên các phiên bản `transformers` của Kaggle.
*   **Chống NaN Loss và Tiết kiệm VRAM**: Sử dụng `GradScaler`, hạ thấp điểm phạt Attention Mask xuống **`-30000.0`**, đặt `eps=1e-6` cho AdamW, và kích hoạt giải phóng bộ nhớ chủ động sau mỗi 10 bước huấn luyện.

## 1. Đồng bộ Mã nguồn trên Kaggle

Bước đầu tiên là thiết lập không gian làm việc cục bộ trên máy ảo Kaggle:
1. Di chuyển thư mục làm việc hiện hành sang `/kaggle/working/` (phần phân vùng có quyền ghi của máy ảo Kaggle).
2. Tiến hành clone kho chứa mã nguồn từ GitHub với nhánh làm việc chính là `develop`.
3. Nếu thư mục dự án đã tồn tại từ trước, hệ thống sẽ tự động chuyển đổi sang nhánh đúng và thực hiện lệnh `git pull` để đồng bộ các cập nhật sửa đổi mới nhất trong thư mục `src/`.

In [ ]:
# =========================================================================
# CELL 1: ĐỒNG BỘ CODE TỪ GITHUB (DEVELOP BRANCH) TRÊN KAGGLE
# =========================================================================
REPO_URL = "https://github.com/dinhtri445/LIVR-Mini-Benchmark.git"
PROJECT_DIR = "LIVR-Mini-Benchmark"
BRANCH = "develop"

%cd /kaggle/working
import os
if not os.path.exists(PROJECT_DIR):
    print(f"---> Đang thực hiện clone repo {REPO_URL} (nhánh {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
    %cd {PROJECT_DIR}
else:
    print(f"---> Repo {PROJECT_DIR} đã tồn tại. Đang tiến hành pull code mới nhất từ nhánh {BRANCH}...")
    %cd {PROJECT_DIR}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}

## 2. Tiền Xử Lý Dữ Liệu & Lọc Trùng Lặp Ảnh Trực Quan

Tập dữ liệu huấn luyện thô chứa nhiều tác vụ khác nhau, trong đó có 4 tác vụ cốt lõi mà mô hình cần học ở giai đoạn 1:
1. **`livr_counting`**: Đếm số lượng thực thể xuất hiện trong hình ảnh.
2. **`livr_object_localization`**: Định vị tọa độ và vị trí tương đối của đối tượng.
3. **`livr_jigsaw`**: Giải các câu đố mảnh ghép hình học trực quan.
4. **`livr_visual_similarity`**: Nhận diện sự tương đồng hoặc khác biệt giữa các họa tiết thị giác.

### 🔍 2.1. Tại sao phải chạy bộ lọc trùng lặp (`filter_and_deduplicate_pipeline`)?
*   **Tránh hiện tượng học tủ (Data Leakage)**: Lọc bỏ các mẫu trùng lặp hoặc quá tương đồng về hình ảnh để đảm bảo mô hình không học vẹt.
*   **Đảm bảo cân bằng tác vụ**: Lấy chính xác số lượng mẫu chỉ định (ví dụ: tối đa 300 mẫu cho mỗi tác vụ) để tạo ra tập dữ liệu huấn luyện cân bằng, tránh việc mô hình bị bias lệch về một tác vụ có nhiều mẫu hơn.

### 💾 2.2. Chiến lược Tối ưu hóa Lưu trữ
*   Sau khi hoàn tất lọc và chuẩn hóa dữ liệu, tập dữ liệu sạch được lưu dưới dạng file serialized của PyTorch (`/kaggle/working/cleaned_dataset.pt`).
*   Ở các phiên chạy sau, code sẽ tự động phát hiện file này để load trực tiếp trong vòng 1-2 giây, loại bỏ hoàn toàn thời gian tải và xử lý dữ liệu thô lặp đi lặp lại.

In [ ]:
# =========================================================================
# CELL 2: CÀI ĐẶT THƯ VIỆN & CHẠY PIPELINE TIỀN XỬ LÝ DỮ LIỆU
# =========================================================================
# Cài đặt các thư viện cần thiết trên môi trường Kaggle
!pip install -r requirements.txt

import sys
import os
import torch

# Đảm bảo Python nhận diện được các module trong thư mục src/
sys.path.append(os.getcwd())

# Cấu hình đường dẫn dữ liệu sạch lưu trên Kaggle working directory để tái sử dụng nhanh
CLEANED_DATA_PATH = "/kaggle/working/cleaned_dataset.pt"

if os.path.exists(CLEANED_DATA_PATH):
    print(f"---> Phát hiện dữ liệu sạch đã được xử lý sẵn tại: {CLEANED_DATA_PATH}")
    print("---> Đang nạp dữ liệu sạch...")
    cleaned_dataset = torch.load(CLEANED_DATA_PATH)
    print(f"[SUCCESS] Đã nạp thành công {len(cleaned_dataset)} mẫu dữ liệu sạch!")
else:
    print("---> Không tìm thấy dữ liệu sạch. Tiến hành tải và làm sạch từ đầu (chỉ chạy 1 lần duy nhất)...")
    from src.utils import load_and_inspect_livr_dataset, filter_and_deduplicate_pipeline
    
    # 1. Nạp và kiểm tra dữ liệu gốc
    dataset = load_and_inspect_livr_dataset()
    
    # 2. Chạy bộ tiền xử lý và khử trùng lặp trực quan (300 mẫu sạch mỗi tác vụ)
    cleaned_dataset = filter_and_deduplicate_pipeline(
        dataset=dataset,
        target_tasks=['livr_counting', 'livr_object_localization', 'livr_jigsaw', 'livr_visual_similarity'],
        samples_per_task=700
    )
    
    # 3. Lưu lại kết quả đã làm sạch để sử dụng lại
    os.makedirs(os.path.dirname(CLEANED_DATA_PATH), exist_ok=True)
    print(f"---> Đang lưu dữ liệu sạch tại: {CLEANED_DATA_PATH}...")
    torch.save(cleaned_dataset, CLEANED_DATA_PATH)
    print("[SUCCESS] Đã lưu dữ liệu sạch thành công!")

## 3. Cấu Hình Kiến Trúc LIVR & Lượng Hóa 4-bit (PEFT LoRA)

Chúng ta tiến hành tải cấu hình từ file `config/implement_config.json` và thực hiện các thiết lập kiến trúc tối ưu cho card GPU của Kaggle:

### 🏎️ 3.1. Lượng hóa 4-bit (NF4 Quantization)
*   Sử dụng cấu hình `load_in_4bit=True` để chuyển đổi trọng số mô hình lớn Qwen2.5-VL về dạng 4-bit thông qua `bitsandbytes`.
*   Kỹ thuật này giúp giảm bộ nhớ VRAM tĩnh từ khoảng 6-7GB xuống chỉ còn **~2GB**, giúp chạy huấn luyện trơn tru mà không sợ OOM.

### 📐 3.2. Cấu hình LoRA (Low-Rank Adaptation)
Thay vì tinh chỉnh toàn bộ 3 tỷ tham số, LoRA chỉ chèn các ma trận phân rã hạng thấp cập nhật trọng số vào các lớp Attention (Q, K, V, O, gate, up, down projections):
$$\Delta W = B \cdot A$$
Với các tham số cấu hình chính:
*   `lora_r`: Hạng (Rank) của ma trận LoRA, thường chọn là 8 hoặc 16.
*   `lora_alpha`: Hệ số tỉ lệ hóa đạo hàm LoRA, thường chọn gấp đôi `r`.
*   `lora_dropout`: Giảm hiện tượng overfitting bằng cách drop ngẫu nhiên các kết nối LoRA.

### 🛠️ 3.3. Bản vá lỗi Monkey-Patch tương thích RoPE và bộ tối ưu AdamW ổn định
*   **Vá lỗi RoPE**: Nhập hàm `patch_model_for_livr` từ `src/mask_kaggle.py`. Hàm này sử dụng `inspect.signature` để đọc động chữ ký hàm của `get_rope_index` và lọc bỏ các tham số không tương thích trên phiên bản `transformers` của Kaggle.
*   **Bộ tối ưu AdamW ổn định**: Thiết lập `eps=1e-6` cho bộ tối ưu AdamW. Việc tăng nhẹ epsilon từ mặc định $10^{-8}$ lên $10^{-6}$ giúp ngăn chặn các phép chia cho số cực kỳ nhỏ, giảm thiểu tối đa rủi ro sinh ra các giá trị vô định dạng **NaN Loss** trong quá trình tính toán float16.

In [ ]:
# =========================================================================
# CELL 3: ĐỌC CONFIG, KHỞI TẠO MÔ HÌNH VÀ BỘ TỐI ƯU (STABILIZED EPS=1e-6)
# =========================================================================
import json
import torch
from torch.optim import AdamW
from src.model import LIVRModelManager
from src.mask_kaggle import patch_model_for_livr

# 1. Đọc file cấu hình định nghĩa sẵn
with open("config/implement_config.json", "r", encoding="utf-8") as f:
    config = json.load(f)

# Thay thế đường dẫn output của Colab bằng thư mục Kaggle
config["output_dir"] = "/kaggle/working/checkpoints"

# Thiết lập các thông số LR và clip ổn định
config['stage1_lr'] = 5e-5
config['stage2_lr'] = 2e-5

print("KAGGLE IMPLEMENTATION CONFIGURATION:")
print(json.dumps(config, indent=2))

# 2. Dựng mô hình LIVR
manager = LIVRModelManager(
    model_id=config["model_id"],
    K=config["K"],
    device="cuda",
    load_in_4bit=config.get("load_in_4bit", True)
)
model = manager.setup_peft_and_freezing(
    r=config["lora_r"],
    alpha=config["lora_alpha"],
    dropout=config["lora_dropout"]
)
processor = manager.processor

# Gọi mask tương thích Kaggle từ file mask_kaggle.py
patch_model_for_livr(
    model=model,
    latent_token_ids=manager.latent_token_ids,
    image_pad_token_id=manager.image_pad_token_id,
    pad_token_id=manager.pad_token_id
)

# 3. Khai báo bộ tối ưu AdamW với eps=1e-6 để chống NaN triệt để
trainable_params = [p for p in model.parameters() if p.requires_grad]
optimizer = AdamW(
    trainable_params, 
    lr=config["stage1_lr"], 
    weight_decay=config["weight_decay"],
    eps=1e-6  
)

print("-> Đã khởi tạo model và Optimizer với eps=1e-6 thành công.")

## 4. Vòng lặp Huấn luyện Hai Giai đoạn (Tích lũy Gradient & GradScaler)

Đây là trái tim của quá trình thực thi huấn luyện. Vòng lặp huấn luyện kiểm soát chặt chẽ luồng đạo hàm và phân bổ bộ nhớ GPU:

### ⚡ 4.1. Giải pháp Tích lũy Gradient (Gradient Accumulation)
*   Do VRAM GPU hạn chế, chúng ta phải đặt batch size thực tế ở mức nhỏ (ví dụ: batch size bằng 1). Tuy nhiên, batch size nhỏ gây ra dao động đạo hàm lớn và huấn luyện kém ổn định.
*   **Gradient Accumulation** giải quyết bài toán này bằng cách chỉ gọi `optimizer.step()` sau khi đã cộng dồn đạo hàm qua `grad_accumulation_steps` (ví dụ: 8 hoặc 16 bước). Điều này giúp giả lập một batch size hiệu dụng lớn hơn gấp nhiều lần mà không làm tăng lượng bộ nhớ VRAM đỉnh.

### 📈 4.2. Chống lỗi Underflow với GradScaler
*   Khi huấn luyện ở dạng nửa chính xác (`float16`), các đạo hàm rất dễ bị rơi vào vùng giá trị cực nhỏ và bị làm tròn về `0.0` (Underflow).
*   **`GradScaler`** sẽ nhân Loss với một hằng số phóng đại lớn. Đạo hàm sau đó được tính toán ở dải giá trị an toàn, rồi được thu nhỏ lại trước khi cập nhật vào tham số mô hình.

### 🛡️ 4.3. Giá trị phạt Attention Mask `-30000.0` tránh NaN
*   Trong quá trình huấn luyện Stage 1, attention mask được sửa đổi để chặn các liên kết trực tiếp giữa văn bản và hình ảnh.
*   Chúng ta sử dụng giá trị phạt **`-30000.0`** thay vì `-65500.0`. Do giới hạn dưới của `float16` là $-65504.0$, giá trị $-30000.0$ mang lại một biên độ an toàn cực lớn cho các phép toán cộng dồn ma trận attention phía sau, loại bỏ hoàn toàn các lỗi tràn số và sinh NaN.

### 🔄 4.4. Cơ chế Tự động Phục hồi và Chuyển đổi Giai đoạn
*   **Tự động tải lại**: Trước khi chạy, code kiểm tra xem đã có checkpoint Stage 1 hợp lệ (kích thước lớn hơn 100KB) được lưu hay chưa. Nếu có, nó sẽ tự động bỏ qua toàn bộ giai đoạn 1, nạp trọng số LoRA và latent embeddings đã học, điều chỉnh Learning Rate xuống mức `stage2_lr` và chuyển thẳng qua huấn luyện Stage 2.
*   **Sửa lỗi checkpoint 66KB**: Trọng số được lưu bằng cách so khớp tên tham số huấn luyện thực tế thông qua danh sách `model.named_parameters()` để đảm bảo file lưu trữ đầy đủ trọng số LoRA và latent embeddings thực tế.

In [ ]:
# =========================================================================
# CELL 4: VÒNG LẶP HUẤN LUYỆN 2 GIAI ĐOẠN (ĐÃ FIX LỖI CHECKPOINT 66KB)
# =========================================================================
import os
import torch
import gc
from tqdm import tqdm
from torch.cuda.amp import GradScaler
from src.utils import prepare_vqa_inputs

# Khởi tạo bộ chia tỷ lệ gradient chống underflow cho float16
scaler = GradScaler()

# Giải phóng bộ nhớ rác trước khi huấn luyện
gc.collect()
torch.cuda.empty_cache()

STAGE1_EPOCHS = config["stage1_epochs"]
STAGE2_EPOCHS = config["stage2_epochs"]
TOTAL_EPOCHS = STAGE1_EPOCHS + STAGE2_EPOCHS
GRADIENT_ACCUMULATION_STEPS = config["grad_accumulation_steps"]
STAGE2_LR = config["stage2_lr"]
checkpoint_dir = config["output_dir"]

# Xác định đường dẫn checkpoint Stage 1
stage1_checkpoint_path = os.path.join(checkpoint_dir, "livr_stage1_checkpoint.pt")

start_epoch = 1

# --- TỰ ĐỘNG LOAD CHECKPOINT STAGE 1 NẾU CÓ ---
if os.path.exists(stage1_checkpoint_path):
    print(f"➔ Phát hiện checkpoint Stage 1 tại: {stage1_checkpoint_path}")
    checkpoint_size_kb = os.path.getsize(stage1_checkpoint_path) / 1024
    
    if checkpoint_size_kb > 100:
        print("➔ Đang tải checkpoint Stage 1 và bỏ qua huấn luyện Stage 1, chuyển trực tiếp sang Stage 2...")
        checkpoint = torch.load(stage1_checkpoint_path, map_location="cuda")
        model.load_state_dict(checkpoint['model_state_dict'], strict=False)
        
        with torch.no_grad():
            model.get_input_embeddings().weight[manager.latent_token_ids].copy_(
                checkpoint['latent_embeddings'].to("cuda")
            )
        
        start_epoch = STAGE1_EPOCHS + 1
        print(f"➔ Cập nhật Learning Rate thành {STAGE2_LR} cho Stage 2...")
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR
            
        del checkpoint
        gc.collect()
        torch.cuda.empty_cache()
    else:
        print("⚠ Cảnh báo: Checkpoint Stage 1 bị lỗi dung lượng (chỉ 66KB). Huấn luyện lại từ đầu...")
else:
    print("➔ Không phát hiện checkpoint Stage 1. Sẽ huấn luyện từ Epoch 1...")

model.train()
epoch_losses = []

print("====== CHÍNH THỨC KHỞI ĐỘNG VÒNG LẶP HUẤN LUYỆN 2 GIAI ĐOẠN (MEM OPTIMIZED) ======")

for epoch in range(start_epoch, TOTAL_EPOCHS + 1):
    current_stage = 1 if epoch <= STAGE1_EPOCHS else 2
    model.livr_stage = current_stage

    if epoch == STAGE1_EPOCHS + 1 and start_epoch <= STAGE1_EPOCHS:
        print(f"\n➔ CHUYỂN GIAI ĐOẠN: Hạ Learning Rate xuống {STAGE2_LR} cho Stage 2...")
        for param_group in optimizer.param_groups:
            param_group['lr'] = STAGE2_LR

    epoch_loss = 0.0
    optimizer.zero_grad()

    progress_bar = tqdm(cleaned_dataset, desc=f"Epoch {epoch}/{TOTAL_EPOCHS} (Stage {current_stage})")

    for step, batch in enumerate(progress_bar):
        try:
            inputs = prepare_vqa_inputs(
                processor=processor,
                conversation=batch['conversation'],
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )

            with torch.amp.autocast('cuda', dtype=torch.float16):
                outputs = model(**inputs)
                loss = outputs.loss / GRADIENT_ACCUMULATION_STEPS

            if torch.isnan(loss):
                print(f"\n[WARNING] Bắt gặp NaN loss tại epoch {epoch}, step {step}. Bỏ qua batch này...")
                optimizer.zero_grad()
                del inputs, outputs, loss
                gc.collect()
                torch.cuda.empty_cache()
                continue
                
            scaler.scale(loss).backward()
            epoch_loss += loss.item() * GRADIENT_ACCUMULATION_STEPS

            if (step + 1) % GRADIENT_ACCUMULATION_STEPS == 0 or (step + 1) == len(cleaned_dataset):
                scaler.unscale_(optimizer)
                torch.nn.utils.clip_grad_norm_(trainable_params, max_norm=0.5)
                scaler.step(optimizer)
                scaler.update()
                optimizer.zero_grad() 

            progress_bar.set_postfix({"Loss": f"{loss.item() * GRADIENT_ACCUMULATION_STEPS:.4f}"})

            del inputs, outputs, loss
            if step % 10 == 0:
                gc.collect()
                torch.cuda.empty_cache()

        except RuntimeError as e:
            if "out of memory" in str(e):
                print("\n[WARNING] Bắt gặp lỗi OOM, đang dọn cache và bỏ qua batch...")
                optimizer.zero_grad()
                del e
                gc.collect()
                torch.cuda.empty_cache()
                continue
            else:
                raise e

    avg_loss = epoch_loss / len(cleaned_dataset)
    epoch_losses.append(avg_loss)
    print(f"➔ Kết thúc Epoch {epoch} - Average Loss tổng thể: {avg_loss:.4f}")

    # --- LƯU CHECKPOINT TRUNG GIAN GIAI ĐOẠN 1 (STAGE 1 CHECKPOINT) ---
    if epoch == STAGE1_EPOCHS:
        os.makedirs(checkpoint_dir, exist_ok=True)
        trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}
        trainable_sd = {k: v.cpu() for k, v in model.state_dict().items() if k in trainable_names}
        
        torch.save({
            'model_state_dict': trainable_sd,
            'latent_embeddings': model.get_input_embeddings().weight[manager.latent_token_ids].detach().cpu(),
            'latent_token_ids': manager.latent_token_ids
        }, stage1_checkpoint_path)
        print(f"---> Đã lưu checkpoint Stage 1 tại: {stage1_checkpoint_path} (Sửa lỗi thành công)")

# --- LƯU CHECKPOINT CUỐI CÙNG (FINAL CHECKPOINT) ---
final_checkpoint_path = os.path.join(checkpoint_dir, "livr_mini_checkpoint.pt")
os.makedirs(checkpoint_dir, exist_ok=True)

trainable_names = {n for n, p in model.named_parameters() if p.requires_grad}
trainable_sd = {k: v.cpu() for k, v in model.state_dict().items() if k in trainable_names}

torch.save({
    'model_state_dict': trainable_sd,
    'latent_embeddings': model.get_input_embeddings().weight[manager.latent_token_ids].detach().cpu(),
    'latent_token_ids': manager.latent_token_ids
}, final_checkpoint_path)

print(f"\n[SUCCESS] Hoàn thành huấn luyện trên Kaggle!")
print(f"Trọng số LoRA + Embeddings đã lưu tại: {final_checkpoint_path}")

## 5. Trực Quan Hóa Biểu Đồ Loss

Chúng ta tiến hành vẽ đường cong Loss theo từng Epoch huấn luyện:
*   Đường nét đứt màu đỏ thể hiện thời điểm chuyển đổi từ **Stage 1 (Bịt mắt)** sang **Stage 2 (Mở mắt)**.
*   Đồ thị này giúp đánh giá trực quan sự hội tụ của mô hình và quan sát xem hành vi của Loss thay đổi như thế nào khi mô hình được mở rộng tầm nhìn để nhìn ảnh trực tiếp.

In [ ]:
# =========================================================================
# CELL 5: TRỰC QUAN HÓA ĐỒ THỊ LOSS
# =========================================================================
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 5))
plt.plot(range(1, len(epoch_losses) + 1), epoch_losses, marker='o', color='b', label='Training Loss')
plt.axvline(x=STAGE1_EPOCHS, color='r', linestyle='--', label='Transition to Stage 2')
plt.title('LIVR-Mini Training Loss Curve (Kaggle)')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.grid(True)
plt.legend()
plt.show()

## 6. Đánh Giá Đối Chiếu Hiệu Năng

Sau khi hoàn tất cả hai giai đoạn huấn luyện, chúng ta tiến hành đánh giá đối chiếu hiệu năng sơ bộ trên 20 mẫu kiểm tra:

### 👁️ 6.1. Chế độ kiểm định:
1. **Mô hình LIVR (Đã học - Lora ON)**: Đánh giá mô hình sau tinh chỉnh tích hợp (Stage 2) xem khả năng suy luận đạt bao nhiêu % độ chính xác.
2. **Mô hình gốc (Hãng - Lora OFF)**: Sử dụng hàm `model.disable_adapter()` để tạm thời ngắt toàn bộ adapter LoRA và latent embeddings. Đưa mô hình trở lại trạng thái nguyên bản của nhà sản xuất.

### 📊 6.2. Ý nghĩa của Baseline Comparison:
*   So sánh độ chính xác giữa 2 chế độ giúp chúng ta đo lường trực tiếp mức độ cải thiện (Performance Boost) mang lại bởi kiến trúc LIVR so với mô hình Qwen gốc.
*   Đây là minh chứng định lượng trực quan nhất thể hiện sự thành công của phương pháp huấn luyện nút cổ chai kết hợp LoRA.

In [ ]:
# =========================================================================
# CELL 6: ĐÁNH GIÁ ĐỐI CHIẾU HIỆU NĂNG BASELINE
# =========================================================================
eval_samples = cleaned_dataset[:20]

def evaluate_baseline(model, processor, manager, samples, use_lora=True):
    if use_lora:
        print("---> Đang đánh giá với LIVR LoRA Adapters (Kích hoạt)...")
        model.eval()
        model.livr_stage = 2
    else:
        print("---> Đang đánh giá với Base Model gốc (Ngắt LoRA)...")
        model.eval()
        
    correct = 0
    total = 0
    log_entries = []
    
    import torch
    from src.utils import prepare_vqa_inputs
    
    with torch.no_grad():
        for i, item in enumerate(samples):
            conv = item['conversation']
            conv_for_generation = [msg for msg in conv if msg["role"] == "user"]
            
            inputs = prepare_vqa_inputs(
                processor=processor,
                conversation=conv_for_generation,
                latent_tokens=manager.latent_tokens,
                device="cuda"
            )
            inputs.pop("labels", None)
            
            if not use_lora:
                with model.disable_adapter():
                    outputs = model.generate(**inputs, max_new_tokens=10)
            else:
                outputs = model.generate(**inputs, max_new_tokens=10)
                
            input_len = inputs["input_ids"].shape[1]
            pred_text = processor.decode(outputs[0][input_len:], skip_special_tokens=True).strip()
            target_text = str(conv[1]["content"][0]["text"]).strip()
            
            is_correct = pred_text.lower() == target_text.lower()
            if is_correct:
                correct += 1
            total += 1
            
            log_entries.append({
                "index": i + 1,
                "target": target_text,
                "predicted": pred_text,
                "correct": is_correct
            })
            
            if i < 5:
                suffix = "LIVR" if use_lora else "Base"
                print(f"   [{suffix} Mẫu {i+1}] Đúng: {target_text} | Đoán: {pred_text} | Kết quả: {'ĐÚNG' if is_correct else 'SAI'}")
                
    accuracy = (correct / total) * 100 if total > 0 else 0.0
    
    # Lưu log chi tiết baseline
    suffix = "livr" if use_lora else "base"
    log_path = os.path.join(config["output_dir"], f"baseline_details_{suffix}.json")
    os.makedirs(os.path.dirname(log_path), exist_ok=True)
    with open(log_path, "w", encoding="utf-8") as lf:
        json.dump(log_entries, lf, ensure_ascii=False, indent=2)
    print(f"   ➔ Đã lưu nhật ký chi tiết baseline ({suffix}) tại: {log_path}")
    
    return accuracy

livr_acc = evaluate_baseline(model, processor, manager, eval_samples, use_lora=True)
print(f"LIVR Model Accuracy: {livr_acc:.2f}%")

base_acc = evaluate_baseline(model, processor, manager, eval_samples, use_lora=False)
print(f"Base Model Accuracy: {base_acc:.2f}%")

print("\n" + "="*50)
print(" KẾT QUẢ ĐỐI CHIẾU HIỆU NĂNG SƠ BỘ (BASELINE COMPARISON)")
print("="*50)
print(f"Mô hình LIVR (Đã học): {livr_acc:.2f}%")
print(f"Mô hình gốc (Hãng):     {base_acc:.2f}%")
print(f"Mức độ cải thiện:        {livr_acc - base_acc:+.2f}%")
print("="*50)